# LeetCode 1779 — Find Nearest Point That Has the Same X or Y Coordinate bằng Python

Link đề: https://leetcode.com/problems/find-nearest-point-that-has-the-same-x-or-y-coordinate/

Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
class Solution:
    def nearestValidPoint(self, x: int, y: int, points: list[list[int]]) -> int:

```

Mục tiêu học:

1. Hiểu **khoảng cách Manhattan** `|x1 - x2| + |y1 - y2|` và hàm `abs`.
2. Áp dụng mẫu **lọc rồi chọn min**, và mẫu **một lần duyệt giữ kỷ lục** (running minimum) kèm **chỉ số**.
3. Xử lý đúng ba yêu cầu phụ: chỉ xét **điểm hợp lệ**, **hòa thì lấy chỉ số nhỏ nhất**, **không có thì trả -1**.
4. Dùng `min(..., key=...)` với **tuple** `(khoảng_cách, chỉ_số)` để tự động phá hòa.
5. Chọn giá trị khởi tạo hợp lý: `float('inf')` hoặc "chưa có" (`-1`).
6. Tự giải biến thể: điểm gần nhất bất kỳ, k điểm gần nhất, khoảng cách Euclid…

## 1. Bài toán yêu cầu gì?

Bạn đứng ở `(x, y)` trên mặt phẳng lưới. Cho mảng `points`, mỗi `points[i] = [a_i, b_i]` là một điểm.

Một điểm là **hợp lệ** nếu nó có **cùng tọa độ x** (`a_i == x`) **hoặc cùng tọa độ y** (`b_i == y`) với bạn.

Trả về **chỉ số** (đếm từ 0) của điểm hợp lệ có **khoảng cách Manhattan nhỏ nhất** tới bạn. Nếu nhiều điểm hòa, trả **chỉ số nhỏ nhất**. Không có điểm hợp lệ → trả `-1`.

Khoảng cách Manhattan giữa `(x1, y1)` và `(x2, y2)` là `abs(x1 - x2) + abs(y1 - y2)`.

| x, y | points | Kết quả | Giải thích |
|---|---|---:|---|
| 3, 4 | `[[1,2],[3,1],[2,4],[2,3],[4,4]]` | 2 | hợp lệ: `[3,1]`(d=3), `[2,4]`(d=1), `[4,4]`(d=1) → hòa, lấy chỉ số 2 |
| 3, 4 | `[[3,4]]` | 0 | điểm trùng vị trí của bạn được phép, d=0 |
| 3, 4 | `[[2,3]]` | -1 | không có điểm hợp lệ |

Constraints: `1 <= points.length <= 10^4`, `points[i].length == 2`, `1 <= x, y, a_i, b_i <= 10^4`.

### Vẽ ví dụ 1 lên lưới

```text
y
5 |
4 |     .  [2]  ★  [4]        ★ = bạn (3,4)
3 |        [3]
2 | [0]
1 |             [1]
  +----------------------- x
     1   2   3   4
```

Điểm hợp lệ nằm trên **đường ngang** `y = 4` hoặc **đường dọc** `x = 3` đi qua `★`.

In [1]:
x, y = 3, 4
points = [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]
for i, (a, b) in enumerate(points):
    valid = a == x or b == y
    print(f"i={i}: ({a},{b}) cùng x? {a == x!s:<5} cùng y? {b == y!s:<5} -> {'HỢP LỆ' if valid else 'loại'}")

i=0: (1,2) cùng x? False cùng y? False -> loại
i=1: (3,1) cùng x? True  cùng y? False -> HỢP LỆ
i=2: (2,4) cùng x? False cùng y? True  -> HỢP LỆ
i=3: (2,3) cùng x? False cùng y? False -> loại
i=4: (4,4) cùng x? False cùng y? True  -> HỢP LỆ


## 2. Nền tảng Python cần biết

### 2.1. `abs()` — giá trị tuyệt đối

`abs(-3) = 3`, `abs(3) = 3`. Khoảng cách luôn không âm dù điểm nằm bên trái hay bên phải.

In [2]:
print(abs(-3), abs(3), abs(0))
print("khoảng cách (3,4)-(2,4):", abs(3 - 2) + abs(4 - 4))
print("khoảng cách (3,4)-(3,1):", abs(3 - 3) + abs(4 - 1))

3 3 0
khoảng cách (3,4)-(2,4): 1
khoảng cách (3,4)-(3,1): 3


### 2.2. Khoảng cách Manhattan vs Euclid

Manhattan đo đường đi "theo ô phố" (chỉ ngang và dọc). Euclid đo đường chim bay `√(dx² + dy²)`.

Với điểm **hợp lệ**, một trong `dx`, `dy` bằng 0 → hai khoảng cách **bằng nhau**! Nhưng đề yêu cầu Manhattan, ta cứ dùng đúng công thức đề.

In [3]:
import math

def manhattan(p, q):
    return abs(p[0] - q[0]) + abs(p[1] - q[1])

def euclid(p, q):
    return math.hypot(p[0] - q[0], p[1] - q[1])

print(manhattan((0, 0), (3, 4)), euclid((0, 0), (3, 4)))   # 7 vs 5.0
print(manhattan((3, 4), (3, 1)), euclid((3, 4), (3, 1)))   # cùng x -> bằng nhau

7 5.0
3 3.0


`math.hypot(dx, dy)` tính `√(dx² + dy²)`.

### 2.3. Toán tử `or`

`A or B` đúng khi **ít nhất một** đúng. Điểm hợp lệ: `a == x or b == y`. Điểm trùng hẳn `(x, y)` thỏa cả hai — vẫn hợp lệ.

In [4]:
print(True or False, False or False, True or True)

True False True


### 2.4. `float('inf')` — "vô cực"

Một giá trị lớn hơn mọi số. Dùng để khởi tạo "khoảng cách tốt nhất" khi chưa gặp điểm nào: điểm hợp lệ đầu tiên chắc chắn nhỏ hơn.

In [5]:
INF = float('inf')
print(INF > 10**18, 5 < INF, min(INF, 7))

True True 7


### 2.5. So sánh tuple

Python so tuple **theo thứ tự từ trái**: so phần tử đầu; nếu bằng mới so phần tử thứ hai. Vì vậy `(1, 2) < (1, 4)` và `(1, 9) < (3, 0)`.

Ứng dụng: `(khoảng_cách, chỉ_số)` — khoảng cách nhỏ hơn thắng; hòa khoảng cách thì chỉ số nhỏ hơn thắng. Đúng luật của đề!

In [6]:
print((1, 2) < (1, 4), (1, 9) < (3, 0), min([(3, 1), (1, 4), (1, 2)]))

True True (1, 2)


### 2.6. `min(iterable, key=..., default=...)`

- `key=f`: so sánh theo `f(phần_tử)` thay vì phần tử.
- `default=v`: nếu iterable **rỗng** thì trả `v` thay vì báo lỗi.

In [7]:
words = ["kiwi", "fig", "banana"]
print(min(words, key=len))
print(min([], default=-1))
try:
    min([])
except ValueError as e:
    print("ValueError:", e)

fig
-1
ValueError: min() iterable argument is empty


## 3. Cách trực giác — lọc điểm hợp lệ, rồi tìm điểm gần nhất

Hai bước, đúng như cách ta đọc đề:

1. **Lọc:** gom các chỉ số `i` có điểm hợp lệ.
2. **Chọn:** trong số đó, tìm khoảng cách nhỏ nhất; hòa thì lấy chỉ số nhỏ nhất. Không có thì `-1`.

In [8]:
def nearest_two_pass(x: int, y: int, points: list[list[int]]) -> int:
    valid = [i for i, (a, b) in enumerate(points) if a == x or b == y]   # bước 1
    if not valid:
        return -1
    best = valid[0]                                                      # bước 2
    for i in valid[1:]:
        if manhattan((x, y), points[i]) < manhattan((x, y), points[best]):
            best = i
    return best

print(nearest_two_pass(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]))  # 2
print(nearest_two_pass(3, 4, [[3, 4]]))                                    # 0
print(nearest_two_pass(3, 4, [[2, 3]]))                                    # -1

2
0
-1


`if not valid:` đúng khi list rỗng — cách viết Python cho "nếu không có phần tử nào".

### 3.1. Cách "ngây thơ" hơn: sắp xếp

Sắp xếp các điểm hợp lệ theo `(khoảng_cách, chỉ_số)` rồi lấy phần tử đầu. Đúng, nhưng sắp xếp tốn `O(n log n)` — thừa, vì ta chỉ cần **một** phần tử nhỏ nhất.

In [9]:
def nearest_sort(x, y, points):
    cands = sorted((manhattan((x, y), p), i) for i, p in enumerate(points) if p[0] == x or p[1] == y)
    return cands[0][1] if cands else -1

print(nearest_sort(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]), nearest_sort(3, 4, [[2, 3]]))

2 -1


## 4. Nút thắt và insight

**Nút thắt:**

- Cách hai bước duyệt dữ liệu **hai lần** và tạo list `valid` tốn `O(n)` bộ nhớ.
- Trong bước 2, mỗi lần so lại **tính lại** khoảng cách của `best`.
- Cách sắp xếp tốn `O(n log n)`.

**Insight (running minimum kèm chỉ số):** chỉ cần **một lần duyệt**, giữ hai biến:

- `best_dist`: khoảng cách nhỏ nhất đã thấy (khởi tạo `inf`).
- `best_idx`: chỉ số tương ứng (khởi tạo `-1` = "chưa có").

Với mỗi điểm hợp lệ, nếu `d < best_dist` (**nhỏ hơn hẳn**) thì cập nhật.

Vì sao `<` mà không `<=`? Ta duyệt chỉ số **tăng dần**; điểm hòa xuất hiện sau có chỉ số lớn hơn → **không** được thay thế. Luật phá hòa được thỏa "miễn phí".

Và nếu không có điểm hợp lệ nào, `best_idx` vẫn là `-1` — đúng yêu cầu, không cần `if` riêng.

In [10]:
# Thí nghiệm: < giữ chỉ số nhỏ nhất, <= lấy chỉ số lớn nhất khi hòa
dists = [(2, 1), (4, 1)]     # (chỉ số, khoảng cách) của 2 điểm hòa trong ví dụ 1
for op in ["<", "<="]:
    best_d, best_i = float('inf'), -1
    for i, d in dists:
        if (d < best_d) if op == "<" else (d <= best_d):
            best_d, best_i = d, i
    print(f"dùng {op:<2}: chọn chỉ số {best_i}")

dùng < : chọn chỉ số 2
dùng <=: chọn chỉ số 4


## 5. Lời giải chính — một lần duyệt, giữ kỷ lục

### 5.1. Khởi tạo

In [11]:
best_dist = float('inf')
best_idx = -1
print(best_dist, best_idx)

inf -1


### 5.2. Xử lý một điểm

Bỏ qua điểm không hợp lệ bằng `continue` (nhảy sang vòng kế tiếp). Điểm hợp lệ: tính `d`, so với kỷ lục.

In [12]:
x, y = 3, 4
for i, (a, b) in enumerate([[1, 2], [2, 4]]):
    if a != x and b != y:
        print(f"i={i}: không hợp lệ -> continue")
        continue
    d = abs(a - x) + abs(b - y)
    if d < best_dist:
        best_dist, best_idx = d, i
    print(f"i={i}: d={d} -> best_dist={best_dist}, best_idx={best_idx}")

i=0: không hợp lệ -> continue
i=1: d=1 -> best_dist=1, best_idx=1


Chú ý điều kiện loại: `a != x and b != y` — phủ định của `a == x or b == y` (luật De Morgan: "không (A hoặc B)" = "không A **và** không B").

### 5.3. Ghép thành hàm

In [13]:
def nearest_valid_point(x: int, y: int, points: list[list[int]]) -> int:
    best_dist = float('inf')
    best_idx = -1
    for i, (a, b) in enumerate(points):
        if a != x and b != y:          # không hợp lệ
            continue
        d = abs(a - x) + abs(b - y)
        if d < best_dist:              # < (không phải <=) để giữ chỉ số nhỏ nhất khi hòa
            best_dist = d
            best_idx = i
    return best_idx

print(nearest_valid_point(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]))
print(nearest_valid_point(3, 4, [[3, 4]]))
print(nearest_valid_point(3, 4, [[2, 3]]))

2
0
-1


## 6. TRACE từng bước

Trace ví dụ 1: `(x, y) = (3, 4)`, `points = [[1,2],[3,1],[2,4],[2,3],[4,4]]`.

| i | điểm | hợp lệ? | d | d < best_dist? | best_dist | best_idx |
|---:|---|---|---:|---|---:|---:|
| — | — | — | — | — | inf | -1 |
| 0 | (1,2) | Không | — | — | inf | -1 |
| 1 | (3,1) | Có (cùng x) | 3 | 3 < inf ✔ | 3 | 1 |
| 2 | (2,4) | Có (cùng y) | 1 | 1 < 3 ✔ | 1 | 2 |
| 3 | (2,3) | Không | — | — | 1 | 2 |
| 4 | (4,4) | Có (cùng y) | 1 | 1 < 1 ✘ (hòa) | 1 | 2 |

Kết quả `2`.

In [14]:
def trace_nearest(x, y, points):
    best_dist, best_idx = float('inf'), -1
    print(f"vị trí của bạn: ({x},{y})")
    for i, (a, b) in enumerate(points):
        if a != x and b != y:
            print(f"  i={i}: ({a},{b}) không hợp lệ -> bỏ qua")
            continue
        d = abs(a - x) + abs(b - y)
        better = d < best_dist
        note = "cập nhật" if better else ("hòa -> giữ chỉ số cũ" if d == best_dist else "xa hơn")
        if better:
            best_dist, best_idx = d, i
        print(f"  i={i}: ({a},{b}) hợp lệ, d={d} -> {note:<20} best=({best_dist}, idx {best_idx})")
    print("=> trả về", best_idx)
    return best_idx

trace_nearest(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]])

vị trí của bạn: (3,4)
  i=0: (1,2) không hợp lệ -> bỏ qua
  i=1: (3,1) hợp lệ, d=3 -> cập nhật             best=(3, idx 1)
  i=2: (2,4) hợp lệ, d=1 -> cập nhật             best=(1, idx 2)
  i=3: (2,3) không hợp lệ -> bỏ qua
  i=4: (4,4) hợp lệ, d=1 -> hòa -> giữ chỉ số cũ best=(1, idx 2)
=> trả về 2


2

### 6.1. Trace khi không có điểm hợp lệ và khi điểm trùng vị trí

In [15]:
trace_nearest(3, 4, [[2, 3], [5, 5]])
print()
trace_nearest(3, 4, [[3, 9], [3, 4], [3, 4]])

vị trí của bạn: (3,4)
  i=0: (2,3) không hợp lệ -> bỏ qua
  i=1: (5,5) không hợp lệ -> bỏ qua
=> trả về -1

vị trí của bạn: (3,4)
  i=0: (3,9) hợp lệ, d=5 -> cập nhật             best=(5, idx 0)
  i=1: (3,4) hợp lệ, d=0 -> cập nhật             best=(0, idx 1)
  i=2: (3,4) hợp lệ, d=0 -> hòa -> giữ chỉ số cũ best=(0, idx 1)
=> trả về 1


1

## 7. Code nộp LeetCode

In [16]:
class Solution:
    def nearestValidPoint(self, x: int, y: int, points: list[list[int]]) -> int:
        best_dist = float('inf')
        best_idx = -1
        for i, (a, b) in enumerate(points):
            if a == x or b == y:
                d = abs(a - x) + abs(b - y)
                if d < best_dist:
                    best_dist = d
                    best_idx = i
        return best_idx

sol = Solution()
print(sol.nearestValidPoint(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]),
      sol.nearestValidPoint(3, 4, [[3, 4]]), sol.nearestValidPoint(3, 4, [[2, 3]]))

2 0 -1


## 8. Các lời giải khác

### 8.1. `min` với key là tuple `(khoảng_cách, chỉ_số)`

Sinh các tuple cho điểm hợp lệ, lấy `min`; tuple tự phá hòa theo chỉ số. `default` xử lý trường hợp rỗng.

In [17]:
class SolutionMinTuple:
    def nearestValidPoint(self, x: int, y: int, points: list[list[int]]) -> int:
        best = min(((abs(a - x) + abs(b - y), i)
                    for i, (a, b) in enumerate(points) if a == x or b == y),
                   default=(None, -1))
        return best[1]

print(SolutionMinTuple().nearestValidPoint(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]),
      SolutionMinTuple().nearestValidPoint(3, 4, [[2, 3]]))

2 -1


### 8.2. `min` trên chỉ số với `key=` là khoảng cách

`min` trả phần tử **đầu tiên** đạt giá trị nhỏ nhất → tự động lấy chỉ số nhỏ nhất khi hòa (vì `range` tăng dần).

In [18]:
class SolutionMinKey:
    def nearestValidPoint(self, x: int, y: int, points: list[list[int]]) -> int:
        valid = [i for i, (a, b) in enumerate(points) if a == x or b == y]
        if not valid:
            return -1
        return min(valid, key=lambda i: abs(points[i][0] - x) + abs(points[i][1] - y))

print(SolutionMinKey().nearestValidPoint(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]))

2


`lambda i: biểu_thức` là **hàm ẩn danh** ngắn: nhận `i`, trả `biểu_thức`. Tương đương `def f(i): return biểu_thức`.

In [19]:
dist_from_origin = lambda p: abs(p[0]) + abs(p[1])
print(dist_from_origin((2, -5)))

7


### 8.3. Mẹo nhỏ: với điểm hợp lệ, khoảng cách = `abs(a - x)` **hoặc** `abs(b - y)`

Nếu `a == x` thì `|a - x| = 0` → `d = |b - y|`. Có thể viết `d = abs(a - x) + abs(b - y)` cho chắc (công thức này luôn đúng), nhưng hiểu điều này giúp thấy bài thực chất là "tìm điểm gần nhất trên hai đường thẳng".

In [20]:
x, y = 3, 4
for a, b in [[3, 1], [2, 4], [4, 4], [3, 4]]:
    print((a, b), "d =", abs(a - x) + abs(b - y), "| |a-x| =", abs(a - x), "| |b-y| =", abs(b - y))

(3, 1) d = 3 | |a-x| = 0 | |b-y| = 3
(2, 4) d = 1 | |a-x| = 1 | |b-y| = 0
(4, 4) d = 1 | |a-x| = 1 | |b-y| = 0
(3, 4) d = 0 | |a-x| = 0 | |b-y| = 0


### 8.4. Sắp xếp — bản nộp (để so sánh)

In [21]:
class SolutionSort:
    def nearestValidPoint(self, x: int, y: int, points: list[list[int]]) -> int:
        cands = sorted((abs(a - x) + abs(b - y), i) for i, (a, b) in enumerate(points) if a == x or b == y)
        return cands[0][1] if cands else -1

print(SolutionSort().nearestValidPoint(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]))

2


### 8.5. Bảng so sánh

Gọi `n = len(points)`, `v` = số điểm hợp lệ.

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---:|---:|---|
| Lọc rồi chọn (2 lần duyệt) | list `valid` | `O(n)` | `O(v)` | Dễ hiểu |
| Sắp xếp | `sorted` theo `(d, i)` | `O(n + v log v)` | `O(v)` | Thừa công |
| Một lần duyệt + kỷ lục | `best_dist`, `best_idx` | `O(n)` | `O(1)` | **Lời giải chính** |
| `min` + tuple | `(d, i)` tự phá hòa | `O(n)` | `O(1)` (generator) | Gọn |
| `min(valid, key=...)` | key là khoảng cách | `O(n)` | `O(v)` | Gọn, dùng `lambda` |

## 9. Vì sao lời giải đúng?

**Bất biến:** sau khi xét các điểm `0..i`, nếu trong đó có điểm hợp lệ thì `best_idx` là **chỉ số nhỏ nhất** trong số các điểm hợp lệ có khoảng cách nhỏ nhất, và `best_dist` là khoảng cách đó; nếu chưa có thì `best_idx = -1`, `best_dist = inf`.

- **Khởi đầu:** chưa xét điểm nào → `-1`, `inf`. Đúng.
- **Điểm `i` không hợp lệ:** không ảnh hưởng tập ứng viên → giữ nguyên. Đúng.
- **Điểm `i` hợp lệ với khoảng cách `d`:**
  - `d < best_dist`: `i` là điểm gần nhất mới duy nhất → cập nhật. Đúng.
  - `d == best_dist`: hòa; ứng viên cũ có chỉ số nhỏ hơn `i` → giữ. Đúng.
  - `d > best_dist`: xa hơn → giữ. Đúng.
- **Kết thúc:** bất biến cho toàn mảng = đúng định nghĩa đáp án (kể cả trường hợp `-1`).

Kiểm chứng bằng máy với cách "sắp xếp theo (d, i)" (rất dễ tin là đúng):

In [22]:
import random

rng = random.Random(1779)
for _ in range(1000):
    x, y = rng.randint(1, 6), rng.randint(1, 6)
    pts = [[rng.randint(1, 6), rng.randint(1, 6)] for _ in range(rng.randint(1, 15))]
    assert nearest_valid_point(x, y, pts) == nearest_sort(x, y, pts) == nearest_two_pass(x, y, pts)
print("1000 bộ ngẫu nhiên (lưới nhỏ, nhiều hòa): một lần duyệt == sắp xếp == hai bước")

1000 bộ ngẫu nhiên (lưới nhỏ, nhiều hòa): một lần duyệt == sắp xếp == hai bước


## 10. Độ phức tạp

Gọi `n = len(points)`.

- **Lời giải chính:** một vòng `n` bước, mỗi bước vài phép so sánh và `abs` → **Time `O(n)`**; chỉ 2 biến → **Space `O(1)`**.
- **Sắp xếp:** `O(n + v log v)` thời gian, `O(v)` bộ nhớ.
- Phải đọc mọi điểm (điểm chưa đọc có thể là điểm trùng vị trí, d = 0) → `O(n)` là tối ưu.

## 11. Bộ test

Nhóm test:

1. Ba ví dụ của đề.
2. Hòa khoảng cách → chỉ số nhỏ nhất (kể cả hòa ở điểm trùng vị trí).
3. Chỉ cùng x / chỉ cùng y.
4. Điểm trùng vị trí xuất hiện sau điểm khác → phải thắng (d = 0).
5. Không điểm hợp lệ với nhiều điểm.
6. `10^4` điểm; đối chiếu các lớp `Solution`.

In [23]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS (={actual})")

sol = Solution()
check("đề 1", sol.nearestValidPoint(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]), 2)
check("đề 2", sol.nearestValidPoint(3, 4, [[3, 4]]), 0)
check("đề 3", sol.nearestValidPoint(3, 4, [[2, 3]]), -1)
check("hòa -> chỉ số nhỏ", sol.nearestValidPoint(5, 5, [[5, 7], [7, 5], [3, 5]]), 0)
check("hòa ở điểm trùng vị trí", sol.nearestValidPoint(1, 1, [[1, 1], [1, 1]]), 0)
check("chỉ cùng x", sol.nearestValidPoint(2, 2, [[2, 9], [2, 5], [8, 8]]), 1)
check("chỉ cùng y", sol.nearestValidPoint(2, 2, [[9, 2], [4, 2], [8, 8]]), 1)
check("trùng vị trí đứng sau thắng", sol.nearestValidPoint(4, 4, [[4, 5], [4, 4]]), 1)
check("không hợp lệ (nhiều điểm)", sol.nearestValidPoint(1, 1, [[2, 2], [3, 3], [4, 4]]), -1)
big = [[10000, 10000]] * 9999 + [[1, 10000]]
check("10^4 điểm", sol.nearestValidPoint(1, 1, big), 9999)

đề 1                               PASS (=2)
đề 2                               PASS (=0)
đề 3                               PASS (=-1)
hòa -> chỉ số nhỏ                  PASS (=0)
hòa ở điểm trùng vị trí            PASS (=0)
chỉ cùng x                         PASS (=1)
chỉ cùng y                         PASS (=1)
trùng vị trí đứng sau thắng        PASS (=1)
không hợp lệ (nhiều điểm)          PASS (=-1)
10^4 điểm                          PASS (=9999)


In [24]:
impls = [Solution(), SolutionMinTuple(), SolutionMinKey(), SolutionSort()]
for _ in range(1000):
    x, y = rng.randint(1, 8), rng.randint(1, 8)
    pts = [[rng.randint(1, 8), rng.randint(1, 8)] for _ in range(rng.randint(1, 20))]
    outs = {s.nearestValidPoint(x, y, pts) for s in impls}
    assert len(outs) == 1, (x, y, pts, outs)
print("1000 bộ ngẫu nhiên: 4 lớp Solution khớp nhau")

1000 bộ ngẫu nhiên: 4 lớp Solution khớp nhau


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` → khi hòa lấy chỉ số **lớn nhất**

In [25]:
def wrong_le(x, y, points):
    best_d, best_i = float('inf'), -1
    for i, (a, b) in enumerate(points):
        if a == x or b == y:
            d = abs(a - x) + abs(b - y)
            if d <= best_d:
                best_d, best_i = d, i
    return best_i

print("sai:", wrong_le(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]), "| đúng: 2")

sai: 4 | đúng: 2


**Sửa:** dùng `<` khi duyệt chỉ số tăng dần.

### Lỗi 2: Dùng `and` thay vì `or` khi kiểm tra hợp lệ

`a == x and b == y` chỉ chấp nhận điểm **trùng hẳn** vị trí.

In [26]:
def wrong_and(x, y, points):
    best_d, best_i = float('inf'), -1
    for i, (a, b) in enumerate(points):
        if a == x and b == y:
            d = 0
            if d < best_d:
                best_d, best_i = d, i
    return best_i

print("sai:", wrong_and(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]), "| đúng: 2")

sai: -1 | đúng: 2


### Lỗi 3: Trả về **khoảng cách** hoặc **tọa độ** thay vì **chỉ số**

Đề hỏi `index`. Trả `best_dist` hay `points[best_idx]` đều sai kiểu.

### Lỗi 4: Khởi tạo `best_dist = 0` (hoặc một số nhỏ)

Không điểm nào có `d < 0` → luôn trả `-1`.

In [27]:
def wrong_init(x, y, points):
    best_d, best_i = 0, -1
    for i, (a, b) in enumerate(points):
        if a == x or b == y:
            d = abs(a - x) + abs(b - y)
            if d < best_d:
                best_d, best_i = d, i
    return best_i

print("sai:", wrong_init(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]), "| đúng: 2")

sai: -1 | đúng: 2


**Sửa:** `float('inf')`, hoặc một số chắc chắn lớn hơn mọi khoảng cách (ở đây tối đa `2·10^4`), ví dụ `10**9`.

### Lỗi 5: Quên dấu `abs`

`(a - x) + (b - y)` có thể âm → điểm "xa phía trái/dưới" lại thành "gần nhất".

In [28]:
def wrong_no_abs(x, y, points):
    best_d, best_i = float('inf'), -1
    for i, (a, b) in enumerate(points):
        if a == x or b == y:
            d = (a - x) + (b - y)
            if d < best_d:
                best_d, best_i = d, i
    return best_i

print("sai:", wrong_no_abs(5, 5, [[6, 5], [1, 5]]), "| đúng:", nearest_valid_point(5, 5, [[6, 5], [1, 5]]))

sai: 1 | đúng: 0


### Lỗi 6: `min(...)` trên generator rỗng không có `default`

Gây `ValueError` (min của dãy rỗng, xem mục 2.6) khi không có điểm hợp lệ. **Sửa:** `default=...` hoặc kiểm tra rỗng trước.

## 13. Mẹo nhận dạng pattern

- **"Tìm phần tử tốt nhất thỏa điều kiện, hòa thì lấy chỉ số nhỏ nhất, không có thì -1"** → một lần duyệt, `best_val = inf`, `best_idx = -1`, cập nhật với `<`.
- Luật phá hòa phức tạp hơn → gói thành **tuple** `(tiêu_chí_1, tiêu_chí_2, ...)` và so tuple.
- "Cùng x **hoặc** cùng y" → `or`; phủ định để `continue` → `and` với `!=`.
- **Manhattan** = tổng `abs` theo từng trục.

Bài cùng họ: 973 K Closest Points to Origin, 1030 Matrix Cells in Distance Order, 1266 Minimum Time Visiting All Points, 2848 Points That Intersect With Cars.

## 14. Biến thể

### 14.1. Điểm gần nhất **bất kỳ** (không cần cùng x/y)

Bỏ điều kiện lọc; khung giữ nguyên.

In [29]:
def nearest_any(x, y, points):
    best_d, best_i = float('inf'), -1
    for i, (a, b) in enumerate(points):
        d = abs(a - x) + abs(b - y)
        if d < best_d:
            best_d, best_i = d, i
    return best_i

print(nearest_any(3, 4, [[1, 2], [3, 1], [2, 3]]))   # (2,3) d=2

2


### 14.2. `k` điểm hợp lệ gần nhất (chỉ số, theo luật phá hòa)

Dùng `heapq.nsmallest(k, iterable)` — lấy `k` phần tử nhỏ nhất mà không cần sắp xếp toàn bộ.

In [30]:
import heapq

def k_nearest_valid(x, y, points, k):
    cands = ((abs(a - x) + abs(b - y), i) for i, (a, b) in enumerate(points) if a == x or b == y)
    return [i for _, i in heapq.nsmallest(k, cands)]

print(k_nearest_valid(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]], 2))   # [2, 4]
print(k_nearest_valid(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]], 5))   # [2, 4, 1]

[2, 4]
[2, 4, 1]


### 14.3. Khoảng cách Chebyshev `max(|dx|, |dy|)`

Đổi công thức khoảng cách là đủ. (Với điểm hợp lệ, Chebyshev cũng bằng Manhattan, vì một trong hai hiệu bằng 0.)

In [31]:
def nearest_any_chebyshev(x, y, points):
    return min(((max(abs(a - x), abs(b - y)), i) for i, (a, b) in enumerate(points)), default=(None, -1))[1]

print(nearest_any_chebyshev(0, 0, [[3, 3], [4, 0], [1, 5]]))   # (3,3)->3, (4,0)->4, (1,5)->5 -> 0

0


## 15. Bài tập tự luyện

Các hàm TODO không được gọi khi chạy notebook. Đáp án ở mục 17.

### Bài 1 — Tính tay

`(x, y) = (2, 3)`, `points = [[2, 8], [5, 3], [1, 1], [2, 1], [0, 3]]`. Điểm nào hợp lệ, khoảng cách bao nhiêu, đáp án là gì?

### Bài 2 — Dự đoán output

`min([(2, 5), (2, 1), (3, 0)])`, `min([], default="rỗng")`, `abs(4 - 9) + abs(7 - 7)` lần lượt ra gì?

### Bài 3 — Điền TODO: một lần duyệt

In [32]:
def exercise_nearest_valid_point(x: int, y: int, points: list[list[int]]) -> int:
    # TODO 1: best_dist = inf, best_idx = -1
    # TODO 2: với mỗi (i, (a, b)): nếu hợp lệ, tính d, cập nhật khi d < best_dist
    # TODO 3: trả best_idx
    pass

# assert exercise_nearest_valid_point(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]) == 2
# assert exercise_nearest_valid_point(3, 4, [[2, 3]]) == -1

### Bài 4 — Tự viết bằng `min` + tuple

In [33]:
def exercise_nearest_min_tuple(x: int, y: int, points: list[list[int]]) -> int:
    # TODO: min(((d, i) for ... if hợp lệ), default=(None, -1))[1]
    pass

# assert exercise_nearest_min_tuple(3, 4, [[3, 4]]) == 0

### Bài 5 — Biến thể: hòa thì lấy chỉ số **lớn nhất**

In [34]:
def exercise_nearest_last_on_tie(x: int, y: int, points: list[list[int]]) -> int:
    # TODO: chỉ cần đổi một ký tự so với lời giải chính
    pass

# assert exercise_nearest_last_on_tie(3, 4, [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]) == 4

### Bài 6 — Mini challenge: nhiều truy vấn

Cho `points` và danh sách truy vấn `queries = [(x1, y1), (x2, y2), ...]`. Trả list đáp án cho từng truy vấn. Cố gắng **gom điểm theo x và theo y** trước (dict `x → list chỉ số`, `y → list chỉ số`) để mỗi truy vấn chỉ xét các điểm cùng x hoặc cùng y, thay vì cả mảng.

In [35]:
def exercise_many_queries(points: list[list[int]], queries: list[tuple[int, int]]) -> list[int]:
    # TODO 1: by_x = {}, by_y = {} (giá trị -> list chỉ số)
    # TODO 2: với mỗi truy vấn, chỉ duyệt by_x.get(x, []) và by_y.get(y, [])
    # TODO 3: chọn theo (d, i) nhỏ nhất; không có thì -1
    pass

# pts = [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]
# assert exercise_many_queries(pts, [(3, 4), (2, 2), (9, 9)]) == [2, 0, -1]

## 16. Gợi ý 3 mức

**Bài 1** — M1: cùng x = 2 hoặc cùng y = 3. M2: hợp lệ: `[2,8]`(d=5), `[5,3]`(d=3), `[2,1]`(d=2), `[0,3]`(d=2). M3: hòa d=2 giữa chỉ số 3 và 4 → **3**.

**Bài 2** — M1: tuple so phần tử đầu trước. M2: `default` dùng khi rỗng. M3: `(2, 1)`, `'rỗng'`, `5`.

**Bài 3** — M1: hai biến kỷ lục. M2: `if a == x or b == y`. M3: `if d < best_dist: best_dist, best_idx = d, i`.

**Bài 4** — M1: generator tuple `(d, i)`. M2: lọc bằng `if` trong generator. M3: lấy `[1]` của kết quả.

**Bài 5** — M1: luật hòa phụ thuộc phép so sánh. M2: `<` → `<=`. M3: chạy lại ví dụ 1 phải ra 4.

**Bài 6** — M1: `by_x.setdefault(a, []).append(i)`. M2: ứng viên = `by_x.get(x, []) + by_y.get(y, [])` (có thể trùng chỉ số — không sao khi lấy min). M3: `min(((dist(i), i) for i in cands), default=(None, -1))[1]`.

## 17. Đáp án tham khảo

**Bài 1:** hợp lệ: chỉ số 0 (d=5), 1 (d=3), 3 (d=2), 4 (d=2) → đáp án **3**.

**Bài 2:** `(2, 1)`, `'rỗng'`, `5`.

In [36]:
trace_nearest(2, 3, [[2, 8], [5, 3], [1, 1], [2, 1], [0, 3]])
print(min([(2, 5), (2, 1), (3, 0)]), min([], default="rỗng"), abs(4 - 9) + abs(7 - 7))

vị trí của bạn: (2,3)
  i=0: (2,8) hợp lệ, d=5 -> cập nhật             best=(5, idx 0)
  i=1: (5,3) hợp lệ, d=3 -> cập nhật             best=(3, idx 1)
  i=2: (1,1) không hợp lệ -> bỏ qua
  i=3: (2,1) hợp lệ, d=2 -> cập nhật             best=(2, idx 3)
  i=4: (0,3) hợp lệ, d=2 -> hòa -> giữ chỉ số cũ best=(2, idx 3)
=> trả về 3
(2, 1) rỗng 5


In [37]:
def answer_nearest_valid_point(x, y, points):
    best_dist, best_idx = float('inf'), -1
    for i, (a, b) in enumerate(points):
        if a == x or b == y:
            d = abs(a - x) + abs(b - y)
            if d < best_dist:
                best_dist, best_idx = d, i
    return best_idx


def answer_nearest_min_tuple(x, y, points):
    return min(((abs(a - x) + abs(b - y), i) for i, (a, b) in enumerate(points) if a == x or b == y),
               default=(None, -1))[1]


def answer_nearest_last_on_tie(x, y, points):
    best_dist, best_idx = float('inf'), -1
    for i, (a, b) in enumerate(points):
        if a == x or b == y:
            d = abs(a - x) + abs(b - y)
            if d <= best_dist:
                best_dist, best_idx = d, i
    return best_idx


def answer_many_queries(points, queries):
    by_x, by_y = {}, {}
    for i, (a, b) in enumerate(points):
        by_x.setdefault(a, []).append(i)
        by_y.setdefault(b, []).append(i)
    res = []
    for x, y in queries:
        cands = by_x.get(x, []) + by_y.get(y, [])
        best = min(((abs(points[i][0] - x) + abs(points[i][1] - y), i) for i in cands), default=(None, -1))
        res.append(best[1])
    return res


pts = [[1, 2], [3, 1], [2, 4], [2, 3], [4, 4]]
assert answer_nearest_valid_point(3, 4, pts) == 2
assert answer_nearest_valid_point(3, 4, [[2, 3]]) == -1
assert answer_nearest_min_tuple(3, 4, [[3, 4]]) == 0
assert answer_nearest_last_on_tie(3, 4, pts) == 4
assert answer_many_queries(pts, [(3, 4), (2, 2), (9, 9)]) == [2, 0, -1]
for _ in range(300):
    P = [[rng.randint(1, 6), rng.randint(1, 6)] for _ in range(rng.randint(1, 12))]
    Q = [(rng.randint(1, 6), rng.randint(1, 6)) for _ in range(5)]
    assert answer_many_queries(P, Q) == [nearest_valid_point(qx, qy, P) for qx, qy in Q]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


`d.setdefault(key, [])` trả `d[key]` nếu đã có, nếu chưa thì gán `d[key] = []` rồi trả list đó — tiện để "thêm vào list theo nhóm".

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def nearestValidPoint(self, x: int, y: int, points: list[list[int]]) -> int:
        best_dist, best_idx = float('inf'), -1
        for i, (a, b) in enumerate(points):
            if a == x or b == y:                       # hợp lệ: cùng x HOẶC cùng y
                d = abs(a - x) + abs(b - y)            # Manhattan
                if d < best_dist:                      # < : hòa giữ chỉ số nhỏ
                    best_dist, best_idx = d, i
        return best_idx                                # -1 nếu không có
```

Template "argmin có điều kiện":

```text
best_val, best_idx = inf, -1
for i, phần_tử in enumerate(dãy):
    if không thỏa điều kiện: continue
    v = giá_trị(phần_tử)
    if v < best_val: best_val, best_idx = v, i
return best_idx
```

Checklist trước khi submit:

- [ ] Điều kiện hợp lệ dùng `or` (cùng x **hoặc** cùng y)?
- [ ] Khoảng cách có `abs` cho cả hai trục?
- [ ] Cập nhật bằng `<` (không `<=`) để hòa lấy chỉ số nhỏ nhất?
- [ ] Khởi tạo `best_dist = inf`, `best_idx = -1`?
- [ ] Trả **chỉ số**, không phải khoảng cách/tọa độ?